In [ ]:
"""
AOI Protocol for Nigerian Major Cities - Modularized Version
"""

# Core imports
import sys
from pathlib import Path

# Add project root and src to Python path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))
sys.path.insert(0, str(project_root / 'src'))

# Import configuration and utilities
import config
from aoi_utils import *
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import folium
from IPython.display import display

# Papermill parameters (for automated execution)
buffer_distance = config.AOI_BUFFER_DISTANCE
output_enabled = True

print("All modules imported successfully!")
print(f"Project root: {config.PROJECT_ROOT}")
print(f"Using buffer distance: {buffer_distance/1000}km")

In [ ]:
# Load all required datasets using centralized function
try:
    cities_df, urban_gdf, boundary_gdf = load_aoi_datasets(
        str(config.get_aoi_data_path('cities_csv')),
        str(config.get_aoi_data_path('urban_geojson')),
        str(config.get_aoi_data_path('boundary_geojson'))
    )
    
    # Validate configuration
    config.validate_config()
    
    print(f"Data loaded successfully!")
    print(f"Available cities: {len(cities_df)}")
    print(f"Urban polygons: {len(urban_gdf)}")
    print(f"Boundary CRS: {boundary_gdf.crs}")
    
except Exception as e:
    print(f"Error loading data: {e}")
    print("Please check your data file paths in config.py")
    
    # Debug information
    print(f"Expected cities CSV: {config.get_aoi_data_path('cities_csv')}")
    print(f"Expected urban GeoJSON: {config.get_aoi_data_path('urban_geojson')}")
    print(f"Expected boundary GeoJSON: {config.get_aoi_data_path('boundary_geojson')}")

In [ ]:
# City selection is controlled via config.py
# CITY_SELECTION_MODE options:
#   - "ALL"      : All 16 cities with population >= 1M (default)
#   - "TEST"     : 5 test cities (Lagos, Abuja, Kano, Jos, Port Harcourt)
#   - "SPECIFIC" : User-specified cities from any of the 50 valid cities

# Get cities to process from config
cities_to_process = config.get_cities_to_process()

# Display configuration status
print(f"City Selection Configuration (from config.py)")
print(f"{'='*60}")
print(f"Current Mode: {config.CITY_SELECTION_MODE}")
print(f"{'='*60}")

# Show mode descriptions
mode_desc = {
    "ALL": f"All cities >= 1M population ({len(config.CITIES_ABOVE_1M)} cities)",
    "TEST": f"Test cities ({len(config.TEST_CITIES)} cities: {', '.join(config.TEST_CITIES)})",
    "SPECIFIC": f"User-specified cities ({len(config.SELECTED_CITIES)} cities)"
}
print(f"Mode Description: {mode_desc.get(config.CITY_SELECTION_MODE, 'Unknown')}")

print(f"\nCities to process ({len(cities_to_process)}):")
for i, city in enumerate(cities_to_process, 1):
    print(f"   {i:2d}. {city}")

# Show other modes info
print(f"\n{'='*60}")
print(f"Available modes (edit CITY_SELECTION_MODE in config.py):")
print(f"  'ALL'      - {len(config.CITIES_ABOVE_1M)} cities >= 1M population")
print(f"  'TEST'     - {len(config.TEST_CITIES)} test cities for quick runs")
print(f"  'SPECIFIC' - Custom selection from {len(config.VALID_CITIES)} valid cities")

In [ ]:
# Process all cities from config-based selection
print(f"Processing {len(cities_to_process)} city/cities: {', '.join(cities_to_process)}\n")

# Dictionary to store results for all cities
all_cities_results = {}

# Loop through each selected city
for idx, current_city in enumerate(cities_to_process, 1):
    print(f"{'='*60}")
    print(f"Processing city {idx}/{len(cities_to_process)}: {current_city}")
    print(f"{'='*60}")
    
    # Process complete AOI using single function call
    aoi_results = process_city_aoi(
        current_city, 
        cities_df, 
        urban_gdf, 
        boundary_gdf,
        buffer_distance=buffer_distance
    )
    
    # Store results
    all_cities_results[current_city] = aoi_results
    
    # Extract results for easier access
    city_info = aoi_results['city_info']
    aoi_gdf = aoi_results['aoi_gdf']
    cities_in_aoi_df = aoi_results['cities_in_aoi_df']
    comprehensive_stats = aoi_results['comprehensive_stats']
    metrics = aoi_results['metrics']
    
    # Display summary
    print(f"\nProcessing Summary for {current_city}:")
    print(f"AOI Area: {metrics['aoi_area_km2']:.2f} km²")
    print(f"Cities in AOI: {len(cities_in_aoi_df)}")
    print(f"Coverage Efficiency: {metrics['coverage_efficiency']:.1f}%")
    if len(cities_in_aoi_df) > 0:
        print(f"Total Population 2024: {comprehensive_stats['total_population_2024']:.0f}k")
        print(f"Major Cities: {comprehensive_stats['AOI_Major_Cities']}")
    print()

print(f"\n{'='*60}")
print(f"Completed processing all {len(cities_to_process)} cities!")
print(f"{'='*60}")

In [ ]:
# Import export functions  
from data_io import export_all_city_data

# Export all data for each processed city
all_export_results = {}

for current_city, aoi_results in all_cities_results.items():
    print(f"\n{'='*60}")
    print(f"Exporting data for: {current_city}")
    print(f"{'='*60}")
    
    # Export all data for current city in the notebook-specific folder
    export_results = export_all_city_data(aoi_results, current_city, "01_aoi_delineation")
    all_export_results[current_city] = export_results
    
    # Display export summary
    print(f"Export Summary for {current_city}:")
    print(f"Output Directory: {export_results['output_directory']}")
    
    # Check which keys actually exist in the export_results
    print("Available keys:", list(export_results.keys()))
    
    # Try to access the directories with fallback
    features_dir = export_results.get('features_directory', 'Not available')
    figures_dir = export_results.get('figures_directory', 'Not available') 
    print(f"Features Directory: {features_dir}")
    print(f"Figures Directory: {figures_dir}")
    print(f"Total Files Exported: {export_results['total_files']}")
    
    # Show exported files by category
    for category, files in export_results['export_paths'].items():
        if isinstance(files, dict):
            print(f"\n{category.replace('_', ' ').title()}:")
            for name, path in files.items():
                filename = Path(path).name
                print(f"  - {name.replace('_', ' ')}: {filename}")
        elif isinstance(files, str):
            filename = Path(files).name
            print(f"{category.replace('_', ' ').title()}: {filename}")

print(f"\n{'='*60}")
print(f"Completed exporting data for all {len(all_cities_results)} cities!")
print(f"{'='*60}")


In [ ]:
# Import visualization functions
from visualization import create_aoi_map, create_interactive_map, create_population_trends_plot, save_figures

print("Creating visualizations for all cities...")

# Create visualizations for each processed city
all_figures_saved = {}

for current_city, aoi_results in all_cities_results.items():
    print(f"\n{'='*60}")
    print(f"Creating visualizations for: {current_city}")
    print(f"{'='*60}")
    
    # Create and display static AOI map
    print("1. Creating comprehensive AOI map...")
    aoi_fig = create_aoi_map(aoi_results, current_city)
    plt.show()
    
    # Create and display interactive map
    print("\n2. Creating interactive map...")
    interactive_map = create_interactive_map(aoi_results, current_city)
    display(interactive_map)
    
    # Create and display population trends plot
    print("\n3. Creating population trends plot...")
    pop_fig = create_population_trends_plot(aoi_results, current_city)
    if pop_fig:
        plt.show()
    
    # Save all figures to the Figures directory
    print("\n4. Saving all figures...")
    export_results = all_export_results[current_city]
    figures_saved = save_figures(aoi_results, current_city, export_results['figures_directory'])
    all_figures_saved[current_city] = figures_saved
    
    print(f"\nVisualization Summary:")
    print(f"Figures Directory: {export_results['figures_directory']}")
    for name, path in figures_saved.items():
        print(f"  - {name}: {Path(path).name}")

print(f"\n{'='*60}")
print(f"Completed visualizations for all {len(all_cities_results)} cities!")
print(f"{'='*60}")
